# Learning Clustering: Understanding Dendrograms

This notebook introduces **hierarchical clustering** and shows how to read a **dendrogram**.

By the end, you should understand:

- What clustering means
- What hierarchical clustering does
- How a dendrogram is built
- How to choose the number of clusters from a dendrogram

## 1. What Is Clustering?

Clustering is an **unsupervised machine learning** technique.

That means we do not give the model labels like `Class A` or `Class B`. Instead, the algorithm tries to find groups based on similarity.

Example:

- Customers with similar buying habits
- Students with similar marks
- Cities with similar weather patterns
- Documents with similar words

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sns.set_theme(style="whitegrid", palette="Set2")

## 3. Create a Simple Dataset

We will create artificial data with visible groups. This makes it easier to understand what the clustering algorithm is doing.

In [ ]:
X, true_labels = make_blobs(
    n_samples=40,
    centers=3,
    cluster_std=1.0,
    random_state=42
)

df = pd.DataFrame(X, columns=["Feature_1", "Feature_2"])
df.head()

## 4. Visualize the Data

Before clustering, always look at the data if it has only two or three features.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["Feature_1"], df["Feature_2"], s=70)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Original Data Points")
plt.show()

## 5. Standardize the Data

Many clustering methods depend on distance. If one feature has much larger values than another, it can dominate the distance calculation.

Standardization puts features on a similar scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

pd.DataFrame(X_scaled, columns=df.columns).head()

## 6. Hierarchical Clustering

Hierarchical clustering builds clusters step by step.

In **agglomerative hierarchical clustering**:

1. Every data point starts as its own cluster.
2. The two closest clusters are merged.
3. This keeps repeating until all points are in one big cluster.

The history of these merges is shown using a **dendrogram**.

## 7. Create the Linkage Matrix

The linkage matrix stores which clusters were merged and at what distance.

Common linkage methods:

- `single`: distance between nearest points
- `complete`: distance between farthest points
- `average`: average distance between points
- `ward`: minimizes within-cluster variance

In [ ]:
linked = linkage(X_scaled, method="ward")

linked[:5]

## 8. Plot the Dendrogram

How to read it:

- Each leaf at the bottom is one data point.
- Branches show clusters being joined.
- The height of a merge shows the distance between clusters.
- A big vertical jump means two very different clusters were merged.

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram(linked)
plt.title("Dendrogram Using Ward Linkage")
plt.xlabel("Data Point Index")
plt.ylabel("Distance")
plt.show()

## 9. Choosing the Number of Clusters

A common method is to look for the **largest vertical gap** in the dendrogram.

Imagine drawing a horizontal line across the dendrogram. The number of vertical branches crossed by the line is the number of clusters.

In the next cell, change `cut_distance` and observe how the result changes.

In [ ]:
cut_distance = 5

plt.figure(figsize=(12, 6))
dendrogram(linked)
plt.axhline(y=cut_distance, color="red", linestyle="--", label=f"Cut distance = {cut_distance}")
plt.title("Dendrogram with Cut Line")
plt.xlabel("Data Point Index")
plt.ylabel("Distance")
plt.legend()
plt.show()

## 10. Assign Cluster Labels

Now we convert the dendrogram cut into actual cluster labels.

In [ ]:
cluster_labels = fcluster(linked, t=cut_distance, criterion="distance")

df_clustered = df.copy()
df_clustered["Cluster"] = cluster_labels

df_clustered.head()

## 11. Visualize the Final Clusters

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(df_clustered["Cluster"].unique()):
    cluster_data = df_clustered[df_clustered["Cluster"] == cluster_id]
    plt.scatter(
        cluster_data["Feature_1"],
        cluster_data["Feature_2"],
        s=80,
        label=f"Cluster {cluster_id}"
    )

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Clusters from Hierarchical Clustering")
plt.legend()
plt.show()

## 12. Try Different Linkage Methods

Run this cell and compare the dendrogram shapes.

In [ ]:
methods = ["single", "complete", "average", "ward"]

plt.figure(figsize=(14, 10))

for i, method in enumerate(methods, start=1):
    linked_method = linkage(X_scaled, method=method)
    plt.subplot(2, 2, i)
    dendrogram(linked_method, no_labels=True)
    plt.title(f"{method.title()} Linkage")
    plt.xlabel("Data Points")
    plt.ylabel("Distance")

plt.tight_layout()
plt.show()

## 13. Mini Exercises for Synthetic Data

Try these changes to strengthen your understanding:

1. Change `cut_distance` to `3`, `4`, `6`, and `8`. How many clusters do you get?
2. Change `n_samples` from `40` to `80`. Does the dendrogram become harder to read?
3. Change `centers=3` to `centers=4`. Can the dendrogram reveal four groups?
4. Compare `single`, `complete`, `average`, and `ward`. Which one gives the clearest dendrogram?

## 14. Real Dataset: Iris Flower Data

Now we will use the downloaded `iris.csv` file. The Iris dataset has measurements for three flower species:

- `setosa`
- `versicolor`
- `virginica`

We will cluster the flowers using only numeric measurements, then compare the clusters with the real species labels.

In [ ]:
iris = pd.read_csv("iris.csv")

print(iris.shape)
iris.head()

## 15. Basic Iris Data Check

Before clustering, inspect the columns, missing values, and species counts.

In [ ]:
iris.info()
print()
print("Missing values:")
print(iris.isna().sum())
print()
print("Species counts:")
print(iris["species"].value_counts())

## 16. Visualize Feature Distributions

Box plots help us see how each flower measurement differs across species.

In [ ]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

plt.figure(figsize=(12, 7))
for i, col in enumerate(feature_cols, start=1):
    plt.subplot(2, 2, i)
    sns.boxplot(data=iris, x="species", y=col)
    plt.title(col.replace("_", " ").title())
    plt.xlabel("")

plt.tight_layout()
plt.show()

## 17. Pair Plot

A pair plot shows relationships between every pair of numeric features. This is useful before clustering because it reveals whether groups are naturally separated.

In [ ]:
sns.pairplot(iris, hue="species", diag_kind="hist", height=2.2)
plt.show()

## 18. Correlation Heatmap

Correlation shows which measurements move together. Petal length and petal width are usually highly related in this dataset.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(iris[feature_cols].corr(), annot=True, cmap="viridis", vmin=-1, vmax=1)
plt.title("Iris Feature Correlation")
plt.show()

## 19. Hierarchical Clustering on Iris

We now remove the `species` label and cluster using only the four numeric measurements.

In [ ]:
X_iris = iris[feature_cols]
y_iris = iris["species"]

iris_scaler = StandardScaler()
X_iris_scaled = iris_scaler.fit_transform(X_iris)

iris_linked = linkage(X_iris_scaled, method="ward")

## 20. Iris Dendrogram

The full Iris dataset has 150 rows, so labels can become crowded. We use `truncate_mode` to make the dendrogram easier to read while preserving the main cluster structure.

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram(
    iris_linked,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=45,
    leaf_font_size=10,
    show_contracted=True
)
plt.title("Truncated Iris Dendrogram")
plt.xlabel("Clustered Samples")
plt.ylabel("Distance")
plt.show()

## 21. Cut the Iris Dendrogram into Clusters

The Iris dataset has three real species, so we ask hierarchical clustering for three clusters and compare them with the known species labels.

In [ ]:
iris_cluster_count = 3
iris_clusters = fcluster(iris_linked, t=iris_cluster_count, criterion="maxclust")

iris_result = iris.copy()
iris_result["cluster"] = iris_clusters

pd.crosstab(iris_result["species"], iris_result["cluster"], rownames=["Actual Species"], colnames=["Cluster"])

## 22. Visualize Iris Clusters

These scatter plots compare real species labels with the clusters found by the algorithm.

In [ ]:
plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(
    data=iris_result,
    x="petal_length",
    y="petal_width",
    hue="species",
    s=80
)
plt.title("Actual Iris Species")

plt.subplot(1, 2, 2)
sns.scatterplot(
    data=iris_result,
    x="petal_length",
    y="petal_width",
    hue="cluster",
    palette="Set1",
    s=80
)
plt.title("Hierarchical Clusters")

plt.tight_layout()
plt.show()

## 23. Add Noise to the Iris Data

Real-world data is rarely perfectly clean. Here we add random noise to the numeric features and observe how clustering changes.

Increase `noise_level` to make the data messier.

In [ ]:
rng = np.random.default_rng(42)
noise_level = 0.45

X_iris_noisy = X_iris_scaled + rng.normal(loc=0, scale=noise_level, size=X_iris_scaled.shape)
noisy_linked = linkage(X_iris_noisy, method="ward")
noisy_clusters = fcluster(noisy_linked, t=3, criterion="maxclust")

iris_noisy_result = iris.copy()
iris_noisy_result["noisy_cluster"] = noisy_clusters

pd.crosstab(
    iris_noisy_result["species"],
    iris_noisy_result["noisy_cluster"],
    rownames=["Actual Species"],
    colnames=["Noisy Cluster"]
)

## 24. Compare Clean vs Noisy Clusters

Noise usually makes cluster boundaries less clear. Compare the two plots below.

In [ ]:
plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(
    x=X_iris_scaled[:, 2],
    y=X_iris_scaled[:, 3],
    hue=iris_clusters,
    palette="Set1",
    s=75
)
plt.title("Clean Scaled Data Clusters")
plt.xlabel("Scaled Petal Length")
plt.ylabel("Scaled Petal Width")

plt.subplot(1, 2, 2)
sns.scatterplot(
    x=X_iris_noisy[:, 2],
    y=X_iris_noisy[:, 3],
    hue=noisy_clusters,
    palette="Set1",
    s=75
)
plt.title(f"Noisy Data Clusters, Noise = {noise_level}")
plt.xlabel("Noisy Scaled Petal Length")
plt.ylabel("Noisy Scaled Petal Width")

plt.tight_layout()
plt.show()

## 25. t-SNE Visualization

`t-SNE` is a dimensionality reduction algorithm. It converts high-dimensional data into two dimensions for visualization.

Important: t-SNE is mainly for visualization, not for proving exact cluster quality.

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    init="pca",
    random_state=42
)

X_tsne = tsne.fit_transform(X_iris_scaled)

tsne_df = pd.DataFrame(X_tsne, columns=["TSNE_1", "TSNE_2"])
tsne_df["species"] = y_iris.values
tsne_df["cluster"] = iris_clusters

tsne_df.head()

## 26. t-SNE: Species vs Clusters

These plots show the same t-SNE projection colored two ways: by actual species and by hierarchical cluster.

In [ ]:
plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(data=tsne_df, x="TSNE_1", y="TSNE_2", hue="species", s=80)
plt.title("t-SNE Colored by Actual Species")

plt.subplot(1, 2, 2)
sns.scatterplot(data=tsne_df, x="TSNE_1", y="TSNE_2", hue="cluster", palette="Set1", s=80)
plt.title("t-SNE Colored by Hierarchical Cluster")

plt.tight_layout()
plt.show()

## 27. t-SNE with Noisy Data

Now compare the clean t-SNE view with a t-SNE view after adding noise.

In [ ]:
tsne_noisy = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    init="pca",
    random_state=42
)

X_tsne_noisy = tsne_noisy.fit_transform(X_iris_noisy)

tsne_noisy_df = pd.DataFrame(X_tsne_noisy, columns=["TSNE_1", "TSNE_2"])
tsne_noisy_df["species"] = y_iris.values
tsne_noisy_df["noisy_cluster"] = noisy_clusters

plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
sns.scatterplot(data=tsne_noisy_df, x="TSNE_1", y="TSNE_2", hue="species", s=80)
plt.title("Noisy t-SNE Colored by Species")

plt.subplot(1, 2, 2)
sns.scatterplot(data=tsne_noisy_df, x="TSNE_1", y="TSNE_2", hue="noisy_cluster", palette="Set1", s=80)
plt.title("Noisy t-SNE Colored by Cluster")

plt.tight_layout()
plt.show()

## 28. Final Exercises

1. Change `noise_level` to `0.1`, `0.7`, and `1.2`. What happens to the clusters?
2. Change t-SNE `perplexity` to `5`, `15`, and `50`. How does the visualization change?
3. Try `complete` or `average` linkage for Iris. Does the crosstab improve or worsen?
4. Use only petal features: `petal_length` and `petal_width`. Is clustering clearer?
5. Use only sepal features: `sepal_length` and `sepal_width`. Is clustering harder?

## 29. Final Key Takeaways

- Iris is a useful real dataset for clustering practice.
- Petal measurements separate species better than sepal measurements.
- Dendrograms show how samples merge as distance increases.
- Adding noise can make clusters overlap and reduce clarity.
- t-SNE helps visualize high-dimensional data in two dimensions, but it should be interpreted carefully.